In [1]:
from sklearn.datasets import make_circles
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split


In [2]:
X,y=make_circles(noise=0.03,n_samples=1000,random_state=42)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [3]:
device=torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

In [4]:
X_train=torch.from_numpy(X_train).type(torch.float32).to(device)
X_test=torch.from_numpy(X_test).type(torch.float32).to(device)
y_test=torch.from_numpy(y_test).type(torch.float32).to(device)
y_train=torch.from_numpy(y_train).type(torch.float32).to(device)

In [5]:
X_train

tensor([[-0.2346, -0.9640],
        [-0.8102,  0.1609],
        [ 0.6811, -0.7357],
        ...,
        [-0.1575,  0.7633],
        [ 0.7784, -0.5890],
        [ 0.6442,  0.8450]], device='cuda:0')

In [36]:
class classficmodel(nn.Module):
    def __init__(self):
        super().__init__()

        self.liner_layer_stack = nn.Sequential(
            nn.Linear(in_features=2, out_features=6),
            nn.Sigmoid(),
            nn.Linear(in_features=6, out_features=6),
            nn.ReLU(),
            nn.Linear(in_features=6, out_features=1)
        )

    def forward(self, x):
        return self.liner_layer_stack(x)

In [37]:
torch.manual_seed(42)
model=classficmodel().to(device)

In [38]:
model

classficmodel(
  (liner_layer_stack): Sequential(
    (0): Linear(in_features=2, out_features=6, bias=True)
    (1): Sigmoid()
    (2): Linear(in_features=6, out_features=6, bias=True)
    (3): ReLU()
    (4): Linear(in_features=6, out_features=1, bias=True)
  )
)

In [39]:
red=model(X_train)
red=red.squeeze(dim=1)
red.shape

torch.Size([800])

In [46]:
loss_fun=nn.L1Loss()
optimizer=torch.optim.Adam(lr=0.01,params=model.parameters())

In [47]:
epoch=3000
for epoch in range(epoch):
    model.train()
    
    train_pred=model(X_train)
    train_pred=train_pred.squeeze(dim=1)
    
    loss=loss_fun(train_pred,y_train)
    
    optimizer.zero_grad()
    
    loss.backward()
    
    optimizer.step()
    
    model.eval()
    
    with torch.inference_mode():
        
        test_pred=model(X_test)
        test_pred=test_pred.squeeze(dim=1)
        
        test_loss=loss_fun(test_pred,y_test)

    if epoch % 100==0:
        
        print(f'epoch -> {epoch}, train loss => {loss.item()} , test_loss-> {test_loss.item()} ')
    

epoch -> 0, train loss => 0.07193256169557571 , test_loss-> 0.10322094708681107 
epoch -> 100, train loss => 0.07167301326990128 , test_loss-> 0.06713702529668808 
epoch -> 200, train loss => 0.07187924534082413 , test_loss-> 0.06635820120573044 
epoch -> 300, train loss => 0.07165252417325974 , test_loss-> 0.06683842837810516 
epoch -> 400, train loss => 0.0726243257522583 , test_loss-> 0.0672668069601059 
epoch -> 500, train loss => 0.07153796404600143 , test_loss-> 0.06728070229291916 
epoch -> 600, train loss => 0.07260998338460922 , test_loss-> 0.06711439788341522 
epoch -> 700, train loss => 0.07145225256681442 , test_loss-> 0.06624840199947357 
epoch -> 800, train loss => 0.0712515339255333 , test_loss-> 0.06539332866668701 
epoch -> 900, train loss => 0.07247524708509445 , test_loss-> 0.06705506145954132 
epoch -> 1000, train loss => 0.07110095769166946 , test_loss-> 0.06577760726213455 
epoch -> 1100, train loss => 0.07077601552009583 , test_loss-> 0.06626440584659576 
epoch -